In [6]:
# SOLAR-10.7B는 약 107억 파라미터 규모
# Upstage Solar API 연결 테스트 (OpenAI SDK 호환)
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("/home/CYJ/.env")  # .env에서 API 키 로드

API_KEY = os.getenv("UPSTAGE_API_KEY")

# base_url만 Upstage로 바꾸면 OpenAI 클라이언트 그대로 사용 가능
client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.upstage.ai/v1"
)

response = client.chat.completions.create(
    model="solar-pro",
    messages=[
        {
            "role": "user",
            "content": "중도해지하면 예금 이자는 어떻게 되는지 쉽게 설명해줘."
        }
    ]
)

print(response.choices[0].message.content)

중도해지 시 예금 이자는 다음과 같이 처리됩니다. 쉽게 정리해 드릴게요!

### 1. **약정 이율 vs 중도해지 이율**
   - 예금은 가입할 때 **약정 이율**로 이자를 주지만, 중도해지 시에는 **훨씬 낮은 이율**이 적용됩니다.  
     (예: 1년 정기예금 약정 이율 4% → 중도해지 시 0.1%~1% 정도)

### 2. **이자 계산 방식**
   - **단리 계산**: 중도해지 이자는 일반적으로 **단리**(원금에 대해서만 이자)로 계산됩니다.  
     (복리보다 이자가 훨씬 적음)
   - **일수 계산**: 해지일까지 실제 예치한 일수만 계산해 이자를 줍니다.  
     (예: 6개월 후 해지 → 6개월분 이자만 지급)

### 3. **세금 영향**
   - 중도해지 이자도 **이자소득세(15.4%)**가 부과됩니다.  
     (단, 이자가 매우 적으면 세금 없을 수 있음)

### 4. **예외 사항**
   - **예금 종류에 따라 다름**:  
     - 일반 정기예금: 중도해지 시 이율 급감  
     - 자유입출금 예금(예: MMDA): 중도해지 페널티 없음  
     - 특별 상품(적금 등): 중도해지 시 이자 전액 없을 수도 있음  
   - **은행별 차이**: 중도해지 이율은 은행마다 다를 수 있으니 약관 확인 필요!

### 📌 **예시**  
- 12개월 정기예금 (약정 이율 4%) → 3개월 후 중도해지  
  - 약정 이자: 100만 원 × 4% × 12개월 = **4만 원**  
  - 중도해지 이자: 100만 원 × 0.2%(은행 정책) × 3개월 ≈ **600원**  
  → 실제 받는 이자는 **극히 적음**

### 💡 **조언**  
- 급한 경우가 아니면 만기까지 유지하는 게 유리합니다.  
- 중도해지가 불가피하면 **이자 손실을 최소화**할 수 있는 상품(유동성 있는 예금)을 선택하세요!  

궁금한 점은 추가로 물어보세요! 😊


In [7]:
# system 프롬프트로 역할을 고정하고 약관 문장을 쉬운 말로 변환
prompt = """
다음 금융 약관을 일반인이 이해하기 쉬운 말로 설명해줘.

[약관]
가입자가 계약기간 만료 전에 계약을 해지하는 경우
중도해지이율을 적용하며, 가입 당시 약정이율은 적용하지 않는다.
"""

response = client.chat.completions.create(
    model="solar-pro",
    messages=[
        {
            "role": "system",
            "content": "너는 금융 약관을 쉽게 설명하는 AI다."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.choices[0].message.content)

"중도해지이율"과 "약정이율"을 쉽게 설명해 드릴게요.  

**▶ 원안:**  
*"가입자가 계약기간 만료 전에 계약을 해지하는 경우 중도해지이율을 적용하며, 가입 당시 약정이율은 적용하지 않는다."*  

**▶ 쉬운 설명:**  
*"만약 만기일 전에 계약을 중도 해지하면, 원래 약속한 금리(약정이율)가 아닌 **더 낮은 중도 해지 특별 금리**를 받게 됩니다. 예를 들어 5년 만기로 연 3% 금리로 가입했더라도, 2년 후 해지하면 1% 정도의 낮은 금리만 적용될 수 있어요. 중도 해지 시 이자 혜택을 전부 받지 못하는 점 유의해 주세요."*

### 추가 설명
- **약정이율**: 가입 시 계약서에 명시된 "원래 받을 예정이었던 금리" (예: 연 3%)  
- **중도해지이율**: 조기 해지할 때 적용되는 "할인된 금리" (예: 연 1%, 약정보다 낮음)  
- **이유**: 금융회사는 장기 계약을 유지하길 원하기 때문에, 중간에 해지하면 불이익을 줍니다.  

예를 들어 예금이나 보험에서 흔히 적용되는 규정이니, 계약 전 중도 해지 가능성을 꼭 고려해 보세요! 😊


In [8]:
# Document Parse API: PDF 상품설명서를 구조화된 요소(텍스트/표 등)로 파싱
import requests

file_path = "/workspace/database/outputs/KBKookmin/Deposit/20260818/pdf/KB 국민ONE통장 상품설명서.pdf"
url = "https://api.upstage.ai/v1/document-digitization"

headers = {
    "Authorization": f"Bearer {API_KEY}"
}

files = {
    "document": open(file_path, "rb")
}

data = {
    "ocr": "force",                  # 텍스트 레이어 무시하고 항상 OCR 수행
    "base64_encoding": "['table']",  # 표 영역은 이미지(base64)도 함께 반환
    "model": "document-parse"
}

response = requests.post(
    url,
    headers=headers,
    files=files,
    data=data
)

result = response.json()

print("요청 완료")
print("status:", response.status_code)

요청 완료
status: 200


In [9]:
# PDF 경로가 실제로 존재하는지 확인
print(os.path.exists("/workspace/database/KBKookmin/Deposit/outputs/pdf/KB 국민ONE통장 상품설명서.pdf"))

False


In [10]:
# 응답 최상위 키 확인 (elements, content, usage 등)
print(result.keys())

dict_keys(['api', 'content', 'elements', 'model', 'ocr', 'usage'])


In [11]:
# 파싱 결과 원본 구조를 일부만 훑어보기
import json
print(json.dumps(result, ensure_ascii=False, indent=2)[:10000])

{
  "api": "2.0",
  "content": {
    "html": "<header id='0' style='font-size:18px'>KB국민은행</header>\n<br><header id='1' style='font-size:18px'>준법감시인 심의필 제2025-0046-1호(심의일:2025.01.13)<br>[유효기간: 2025.01.13~ 2027.01.10]</header>\n<header id='2' style='font-size:14px'>Kb KB 국민은행</header>\n<br><h1 id='3' style='font-size:22px'>「KB 국민ONE통장」 상품설명서</h1>\n<p id='4' data-category='paragraph' style='font-size:20px'>◈ 이 설명서는 금융소비자의 권익 보호 및 예금상품에 대한 이해 증진을 위하여 「금융소비자 보호에 관한 법률」 및<br>관련 규정에 의거, 은행의 내부 통제절차를 거쳐 예금상품의 주요 내용을 쉽게 이해할 수 있도록 작성한<br>자료입니다.</p>\n<br><p id='5' data-category='list' style='font-size:20px'>◈ 설명내용을 제대로 이해하지 못하였음에도 불구하고 설명을 이해했다는 서명을 하거나 녹취기록을 남기시는 경우,<br>추후 해당 내용과 관련한 권리구제가 어려울 수 있습니다.<br>◈ 인터넷뱅킹 등 비대면채널을 통해 가입하거나 통장 미발행 요청 시에는 통장 또는 증서가 교부되지 않습니다.</p>\n<h1 id='6' style='font-size:20px'>1 상품명 및 특징</h1>\n<p id='7' data-category='list' style='font-size:16px'>· 상 품 명 : KB국민ONE통장<br>· 상품특징 : 생활 속 은행 거래실적에 따라 다양한 수수료 면제 서비스를 제공하는 입출금이 자유로운 예금</p>\n<h1 id='8' style='font-size:20px'>2 

In [12]:
# 요소 카테고리 분포 확인 (paragraph / table / heading ...)
from collections import Counter

print(Counter(
    element["category"]
    for element in result["elements"]
))

Counter({'table': 5, 'paragraph': 4, 'header': 3, 'heading1': 3, 'list': 3})


In [13]:
# 표(table) 요소만 골라 HTML로 출력
for element in result["elements"]:
    if element["category"] == "table":
        print(element["content"]["html"])
        print("=" * 80)

<table id='9' style='font-size:18px'><thead><tr><td>구 분</td><td>내 용</td></tr></thead><tbody><tr><td>가입대상</td><td>· 실명의 개인(1 인 1 계좌) ※ 개인사업자 및 관련서류 미제출 임의단체 포함</td></tr><tr><td>신규금액</td><td>· 제한없음</td></tr><tr><td>상품유형</td><td>· 입출금이 자유로운 예금(보통예금, 저축예금) ※ 동일 예금과목 상품 전환가능 ※ 개별상품에서 정한 기준에 의해 전환신규가 불가한 상품은 제외</td></tr><tr><td>기본이율 (2025.01.13 기준, 세금공제 전)</td><td>· 연 0.10% ※ 결산일 현재 영업점에 고시한 예금과목별 기본이율 적용</td></tr><tr><td>이자 지급시기</td><td>· 보통예금 : 이자는 매년 6, 12월의 둘째 금요일에 결산하여 다음날 원금에 더함 · 저축예금 : 이자는 매년 2, 5, 8, 11월의 둘째 금요일에 결산하여 다음날 원금에 더함 ※ 예금의 이자는 최종거래일로부터 5년이 경과한 날부터는 이자를 원금에 더하지 않고, 계좌 해지 또는 추가 입출금 거래 발생일에 일괄 계산하여 지급할 수 있습니다.</td></tr><tr><td>이자 계산방법 (산출근거)</td><td>· 예금이자는 예금일부터 지급일 전일까지(결산시에는 지난 원가일부터 원가일 전일까지)를 이자계산 기간으로 하고, 매일의 최종잔액을 평균하여 결산일 현재 영업점에 고시하는 이율로 계산함</td></tr></tbody></table>
<table id='10' style='font-size:14px'><thead></thead><tbody><tr><td>KB생활 서비스란?</td><td>· KB생활서비스는 KB국민ONE통장 가입고객에게 제공하는 수수료면제 서비스입니다. ※ 실적 발생 및 수수료면제는 이 통장 기준이며, 수수료면제 잔여횟수는 이월되지 않습니다. ※ 예시는 참고 사항이므로 자

In [14]:
# 전체 요소를 순서대로 이어붙여 문서 하나로 재구성
# (카테고리를 HTML 주석으로 남겨 나중에 청킹/필터링에 활용)
md_text = ""

for element in result["elements"]:
    category = element["category"]
    html = element["content"]["html"]

    md_text += f"\n<!-- {category} -->\n"
    md_text += html
    md_text += "\n"

In [15]:
# 재구성 결과 확인
print(md_text[:5000])


<!-- header -->
<header id='0' style='font-size:18px'>KB국민은행</header>

<!-- header -->
<br><header id='1' style='font-size:18px'>준법감시인 심의필 제2025-0046-1호(심의일:2025.01.13)<br>[유효기간: 2025.01.13~ 2027.01.10]</header>

<!-- header -->
<header id='2' style='font-size:14px'>Kb KB 국민은행</header>

<!-- heading1 -->
<br><h1 id='3' style='font-size:22px'>「KB 국민ONE통장」 상품설명서</h1>

<!-- paragraph -->
<p id='4' data-category='paragraph' style='font-size:20px'>◈ 이 설명서는 금융소비자의 권익 보호 및 예금상품에 대한 이해 증진을 위하여 「금융소비자 보호에 관한 법률」 및<br>관련 규정에 의거, 은행의 내부 통제절차를 거쳐 예금상품의 주요 내용을 쉽게 이해할 수 있도록 작성한<br>자료입니다.</p>

<!-- list -->
<br><p id='5' data-category='list' style='font-size:20px'>◈ 설명내용을 제대로 이해하지 못하였음에도 불구하고 설명을 이해했다는 서명을 하거나 녹취기록을 남기시는 경우,<br>추후 해당 내용과 관련한 권리구제가 어려울 수 있습니다.<br>◈ 인터넷뱅킹 등 비대면채널을 통해 가입하거나 통장 미발행 요청 시에는 통장 또는 증서가 교부되지 않습니다.</p>

<!-- heading1 -->
<h1 id='6' style='font-size:20px'>1 상품명 및 특징</h1>

<!-- list -->
<p id='7' data-category='list' style='font-size:16px'>· 상 품 명 : KB국민ONE통장<br>· 상품특징 : 생활 속 은